In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import scraping_functions as sf
import importlib
import time
import re

importlib.reload(sf);

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [3]:
def get_page(url, headers, retries=5):
    for attempt in range(retries):
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            return response

        print(f"Attempt {attempt+1}: {response.status_code}")
        time.sleep(3)

    return response

In [4]:
url = f'https://www.transfermarkt.com/premier-league/spieltag/wettbewerb/GB1/saison_id/2025/spieltag/1'
response = get_page(url, headers)
soup = BeautifulSoup(response.content, "lxml")

## Matches csv

In [5]:
all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

# SCRAPING AWAY INFORMATION
print('-----------------------------------------')
print('SCRAPING AWAY INFORMATION')
print('-----------------------------------------')

away_team_info = all_matches[5].find_all('td',{'class':away_team})

away_team_url = away_team_info[0].find('a').get('href')
away_team_id = away_team_url.split('/')[-3]
away_team_name = away_team_info[0].find('a').get('title')

display(away_team_info,away_team_url,away_team_id,away_team_name)

# SCRAPING HOME INFORMATION
print('-----------------------------------------')
print('SCRAPING HOME INFORMATION')
print('-----------------------------------------')

home_team_info = all_matches[5].find_all('td',{'class':home_team})

home_team_url = home_team_info[0].find('a').get('href')
home_team_id = home_team_url.split('/')[-3]
home_team_name = home_team_info[0].find('a').get('title')

display(home_team_info,home_team_url,home_team_id,home_team_name)

-----------------------------------------
SCRAPING AWAY INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen">
 <a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City"><img alt="Manchester City" class="" src="https://img.a.transfermarkt.technology/wappen/small/281.png?lm=4711" title="Manchester City"/></a> </td>]

'/manchester-city/spielplan/verein/281/saison_id/2025'

'281'

'Manchester City'

-----------------------------------------
SCRAPING HOME INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen">
 <a href="/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025" title="Wolverhampton Wanderers"><img alt="Wolverhampton Wanderers" class="" src="https://img.a.transfermarkt.technology/wappen/small/543.png?lm=4711" title="Wolverhampton Wanderers"/></a> </td>]

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

In [90]:
match_url = all_matches[0].find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

match_id = match_url.split('/')[-1]

match_result = all_matches[0].find('span',{'class':'matchresult finished'}).string

match_info = all_matches[0].find_all('td',{'class':'zentriert no-border'})

match_day = match_info[0].find('a').get('href').split('/')[-1]
match_referee = match_info[1].find('a').string
match_attendance = match_info[2].get_text().strip()
match_attendance = re.sub('[.]','', match_attendance).split(' ')[0]
time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
match_time = time_info[0]
match_time_period = time_info[-1]

display(match_info,match_day,match_referee,match_attendance,match_time,match_time_period,match_result,match_url,match_id)


[<td class="zentriert no-border" colspan="5">
 <div class="di"><span class="hide-for-small">Friday,</span><span class="show-for-small">Fri</span></div> <a href="/aktuell/waspassiertheute/aktuell/new/datum/2025-08-15">
                                                                 15/08/2025                                        </a>
                                                              - 9:00 PM
                             </td>,
 <td class="zentriert no-border" colspan="5">
 <span>Referee: <a href="/anthony-taylor/profil/schiedsrichter/847" title="Anthony Taylor">Anthony Taylor</a></span> </td>,
 <td class="zentriert no-border" colspan="5">
 <span class="icons_sprite icon-zuschauer-zahl" title="Attendance"> </span>
                                         60.315                                </td>]

'2025-08-15'

'Anthony Taylor'

'60315'

'9:00'

'PM'

'4:2'

'/spielbericht/index/spielbericht/4625774'

'4625774'

In [55]:
display(home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period)

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

'4:2'

'/manchester-city/spielplan/verein/281/saison_id/2025'

...

'15/08/2025'

'Anthony Taylor'

'60315'

'9:00'

'PM'

## Events csv

In [41]:
match_event = all_matches[0].find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})

display(match_event)

[<tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">Hugo Ekitiké</a></span></div><div class="di nowrap"><span class="show-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">H. Ekitiké</a></span></div></div><span class="icons_sprite icon-tor-formation" title="Minute 37: Goal"> </span></td>
 <td class="zentriert no-border-links">37'</td>
 <td class="zentriert hauptlink">1:0</td>
 <td class="zentriert no-border-rechts"> </td>
 <td class="links no-border-links"> </td>
 </tr>,
 <tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/cody-gakpo/profil/spieler/434675" title="Cody Gakpo">Cody Gakpo</a></span></div><div class="di nowrap"><span class="show-for-sm

In [42]:
player_url = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('href')
player_id = player_url.split('/')[-1]
player_name = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('title')

event_type = match_event[0].find('span',{'class':'icons_sprite'}).get('class')[-1]

check = match_event[0].find('td',{'class':'zentriert hauptlink'})
event_score = None if check == None else check.string

home='links'
away='rechts'

check = lambda x: match_event[0].find('td',{'class':f'zentriert no-border-{x}'}).string
event_time_label = check(away) if check(home) == '\xa0' else check(home)

time_list = re.sub("[']",'', event_time_label).split('+')
event_time_minute = int(time_list[0])
event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

display(player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra)

'/hugo-ekitike/profil/spieler/709726'

'709726'

'Hugo Ekitiké'

'icon-tor-formation'

'1:0'

"37'"

37

0

In [58]:
all_leagues = {
	'premier-league' : 'GB1',
	'bundesliga' : 'L1',
	'serie-a' : 'IT1',
	'laliga' : 'ES1',
	'ligue-1' : 'FR1',
	'campeonato-brasileiro-serie-a' : 'BRA1'
}

In [95]:
def get_events(headers, league, n_season, n_round):
    # Creating soup object
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Gathering all 10 matches from the round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})
    season_id = f'{all_leagues[league]}-{n_season}'

    output_list = []
    # separating matches
    for m, match in enumerate(all_matches):
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'
        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        # separating events
        match_event = match.find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})
        for event in match_event:
            temp = []

            # Player Information
            player_url = event.find('td',{'class':'spieltagsansicht'}).find('a').get('href')
            player_id = int(player_url.split('/')[-1])
            player_name = event.find('td',{'class':'spieltagsansicht'}).find('a').get('title')

            #Event Information
            event_type = event.find('span',{'class':'icons_sprite'}).get('class')[-1]
            check = event.find('td',{'class':'zentriert hauptlink'})
            event_score = None if check == None else check.string

            # Time Information
            home='links'
            away='rechts'

            check = lambda x: event.find('td',{'class':f'zentriert no-border-{x}'}).string
            event_time_label = check(away) if check(home) == '\xa0' else check(home)

            time_list = re.sub("[']",'', event_time_label).split('+')
            event_time_minute = int(time_list[0])
            event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

            # Appending to the output
            temp.append(season_id)
            temp.append(match_id)

            temp.append(match_url)
            temp.append(player_url)
            temp.append(player_id)
            temp.append(player_name)
            temp.append(event_type)
            temp.append(event_score)
            temp.append(event_time_label)
            temp.append(event_time_minute)
            temp.append(event_time_extra)

            output_list.append(temp)
    output_list.insert(0,['season_id','match_id','match_url','player_url','player_id','player_name','event_type','event_score','event_time_label','event_time_minute','event_time_extra'])
    return output_list

In [96]:
ltest = get_events(headers,'premier-league',2025,1)

df = pd.DataFrame(ltest[1:],columns=ltest[0])

display(df)

,season_id,match_id,match_url,player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/hugo-ekitike/profil/spieler/709726,709726,Hugo Ekitiké,icon-tor-formation,1:0,37',37,0
1,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/cody-gakpo/profil/spieler/434675,434675,Cody Gakpo,icon-tor-formation,2:0,49',49,0
2,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:1,64',64,0
3,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:2,76',76,0
4,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/federico-chiesa/profil/spieler/341092,341092,Federico Chiesa,icon-tor-formation,3:2,88',88,0
5,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/mohamed-salah/profil/spieler/148455,148455,Mohamed Salah,icon-tor-formation,4:2,90+4',90,4
6,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/ezri-konsa/profil/spieler/413403,413403,Ezri Konsa,icon-rotekarte-formation,None,66',66,0
7,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/matt-oriley/profil/spieler/406634,406634,Matt O'Riley,icon-elfmeter-formation,1:0,55',55,0
8,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/rodrigo-muniz/profil/spieler/735571,735571,Rodrigo Muniz,icon-tor-formation,1:1,90+6',90,6
9,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/eliezer-mayenda/profil/spieler/967346,967346,Eliezer Mayenda,icon-tor-formation,1:0,61',61,0


In [91]:
def get_matches(headers, league, n_season, n_round):
    # Creating soup object
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Gathering all 10 matches from the round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

    # Different calls for home and away teams
    home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
    away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

    season_id = f'{all_leagues[league]}-{n_season}'

    output_list = []
    # separating matches
    for m, match in enumerate(all_matches):
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'

        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        temp = []
        # Information from each match
        # Scraping Away Information
        away_team_info = match.find_all('td',{'class':away_team})

        away_team_url = away_team_info[0].find('a').get('href')
        away_team_id = away_team_url.split('/')[-3]
        away_team_name = away_team_info[0].find('a').get('title')

        # Scraping Home Information
        home_team_info = match.find_all('td',{'class':home_team})

        home_team_url = home_team_info[0].find('a').get('href')
        home_team_id = home_team_url.split('/')[-3]
        home_team_name = home_team_info[0].find('a').get('title')

        # Scraping Result
        match_result = match.find('span',{'class':'matchresult finished'}).string

        # Scraping Addicional Information
        match_info = match.find_all('td',{'class':'zentriert no-border'})

        match_day = match_info[0].find('a').get('href').split('/')[-1]
        match_referee = match_info[1].find('a').string
        match_attendance = match_info[2].get_text().strip()
        match_attendance = re.sub('[.]','', match_attendance).split(' ')[0]
        time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
        match_time = time_info[0]
        match_time_period = time_info[-1]

        # Appending to the output
        temp.append(season_id)
        temp.append(match_id)

        temp.append(match_url)
        temp.append(home_team_url)
        temp.append(home_team_id)
        temp.append(home_team_name)
        temp.append(match_result)
        temp.append(away_team_url)
        temp.append(away_team_id)
        temp.append(away_team_name)
        temp.append(match_day)
        temp.append(match_referee)
        temp.append(match_attendance)
        temp.append(match_time)
        temp.append(match_time_period)

        output_list.append(temp)
    return output_list

In [92]:
ltest2 = get_matches(headers,'premier-league',2025,1)

df = pd.DataFrame(ltest2)

display(df)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,4:2,/afc-bournemouth/spielplan/verein/989/saison_i...,989,AFC Bournemouth,2025-08-15,Anthony Taylor,60315,9:00,PM
1,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/aston-villa/spielplan/verein/405/saison_id/2025,405,Aston Villa,0:0,/newcastle-united/spielplan/verein/762/saison_...,762,Newcastle United,2025-08-16,Craig Pawson,42526,1:30,PM
2,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,1:1,/fc-fulham/spielplan/verein/931/saison_id/2025,931,Fulham FC,2025-08-16,Samuel Barrott,31478,4:00,PM
3,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,3:0,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,2025-08-16,Robert Jones,46233,4:00,PM
4,GB1-2025,M-2025-01-05,/spielbericht/index/spielbericht/4625779,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,3:0,/fc-burnley/spielplan/verein/1132/saison_id/2025,1132,Burnley FC,2025-08-16,Michael Oliver,61077,4:00,PM
5,GB1-2025,M-2025-01-06,/spielbericht/index/spielbericht/4625780,/wolverhampton-wanderers/spielplan/verein/543/...,543,Wolverhampton Wanderers,0:4,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,2025-08-16,Jarred Gillett,31118,6:30,PM
6,GB1-2025,M-2025-01-07,/spielbericht/index/spielbericht/4625776,/nottingham-forest/spielplan/verein/703/saison...,703,Nottingham Forest,3:1,/fc-brentford/spielplan/verein/1148/saison_id/...,1148,Brentford FC,2025-08-17,Peter Bankes,29949,3:00,PM
7,GB1-2025,M-2025-01-08,/spielbericht/index/spielbericht/4625781,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,0:0,/crystal-palace/spielplan/verein/873/saison_id...,873,Crystal Palace,2025-08-17,Darren England,39678,3:00,PM
8,GB1-2025,M-2025-01-09,/spielbericht/index/spielbericht/4625782,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,0:1,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,2025-08-17,Simon Hooper,73475,5:30,PM
9,GB1-2025,M-2025-01-10,/spielbericht/index/spielbericht/4625783,/leeds-united/spielplan/verein/399/saison_id/2025,399,Leeds United,1:0,/fc-everton/spielplan/verein/29/saison_id/2025,29,Everton FC,2025-08-18,Chris Kavanagh,36820,9:00,PM
